In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2005_2006"
    / "Labs"
)

xpt_file = DATA_DIR / "GHB_D.xpt"
csv_file = DATA_DIR / "GHB_D_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Keep untouched source for fidelity comparison
source_df = df.copy()


print("=" * 80)
print("NHANES 2005-2006 GHB_D XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. STRUCTURE CHECK
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "LBXGH"
]


if len(df) != 6980:
    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 2:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "GHB_D columns/order differ.\n"
        f"Actual: {df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: GHB_D structure = "
    "6,980 rows x 2 columns."
)


# ============================================================
# 4. CHECK XPORT TINY-VALUE ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


print("\n--- TINY VALUE CHECK ---")

print(
    "Tiny artifact occurrences:",
    tiny_total
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
    )

    raise ValueError(
        "STOP: GHB_D unexpectedly contains "
        "tiny XPORT values. "
        "Do NOT automatically replace them."
    )


print(
    "PASS: No tiny-value artifact exists."
)


# ============================================================
# 5. TRUE ZERO CHECK
# ============================================================

zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


zero_total = int(
    zero_counts.sum()
)


print(
    "\nTrue numeric zeros:",
    zero_total
)


if zero_total != 0:

    print(
        zero_counts[
            zero_counts > 0
        ]
    )

    raise ValueError(
        "Unexpected numeric zeros found. "
        "Review before proceeding."
    )


print(
    "PASS: No numeric zero values in source."
)


# ============================================================
# 6. SEQN VALIDATION
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


print("\n--- SEQN ---")

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(df["SEQN"].isna().sum()) != 0
    or
    int(df["SEQN"].duplicated().sum()) != 0
    or
    df["SEQN"].min() != 31129
    or
    df["SEQN"].max() != 41474
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 7. LBXGH LABORATORY MEASUREMENT CHECK
# ============================================================

#
# CRITICAL:
#
# LBXGH is a laboratory measurement.
#
# DO NOT:
# - convert LBXGH to Int64
# - round LBXGH
# - recalculate LBXGH
# - trim legitimate .0
# - apply cross-over equations
#

s = df["LBXGH"]


print("\n--- LBXGH ---")

print(
    "Valid:",
    int(s.notna().sum())
)

print(
    "Missing:",
    int(s.isna().sum())
)

print(
    "Minimum:",
    s.min()
)

print(
    "Maximum:",
    s.max()
)


if int(
    s.notna().sum()
) != 6493:

    raise ValueError(
        "LBXGH valid count differs from CDC."
    )


if int(
    s.isna().sum()
) != 487:

    raise ValueError(
        "LBXGH missing count differs from CDC."
    )


if not np.isclose(
    s.min(),
    3.8,
    rtol=0,
    atol=0
):

    raise ValueError(
        "LBXGH minimum differs from CDC."
    )


if not np.isclose(
    s.max(),
    15.6,
    rtol=0,
    atol=0
):

    raise ValueError(
        "LBXGH maximum differs from CDC."
    )


print(
    "PASS: LBXGH matches CDC "
    "(3.8-15.6, 6,493 valid, 487 missing)."
)


# ============================================================
# 8. VERIFY LBXGH CONTAINS REAL DECIMALS
# ============================================================

lbxgh_values = (
    df["LBXGH"]
    .dropna()
)


fractional = ~np.isclose(
    lbxgh_values,
    np.round(lbxgh_values),
    rtol=0,
    atol=1e-12
)


fractional_count = int(
    fractional.sum()
)


print(
    "\nLBXGH genuine fractional values:",
    fractional_count
)


if fractional_count == 0:

    raise ValueError(
        "Unexpected: LBXGH contains no fractional values."
    )


print(
    "PASS: LBXGH is correctly treated "
    "as a continuous laboratory measurement."
)


# ============================================================
# 9. CLEAN ONLY SEQN
# ============================================================

#
# SEQN:
# 31129.0 -> 31129
#
# LBXGH remains untouched.
#

df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print(
    "\nPASS: SEQN converted to integer format."
)

print(
    "LBXGH remains unchanged."
)


# ============================================================
# 10. VALUE FIDELITY BEFORE EXPORT
# ============================================================

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


# SEQN numeric values must remain identical
seqn_after = (
    df["SEQN"]
    .astype("Float64")
    .to_numpy(
        dtype=float,
        na_value=np.nan
    )
)


if not np.allclose(
    source_df["SEQN"].to_numpy(),
    seqn_after,
    rtol=0,
    atol=0,
    equal_nan=True
):

    raise ValueError(
        "SEQN values changed."
    )


# LBXGH must be bit-for-bit/numerically unchanged
if not np.allclose(
    source_df["LBXGH"].to_numpy(),
    df["LBXGH"].to_numpy(),
    rtol=0,
    atol=0,
    equal_nan=True
):

    raise ValueError(
        "LBXGH values changed."
    )


print(
    "PASS: SEQN numeric values unchanged."
)

print(
    "PASS: LBXGH values unchanged."
)


# ============================================================
# 11. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO float_format
# NO round()
# NO recalculation
#
# pandas will preserve strings such as:
#
# 5.2
# 6.0
# 15.6
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 12. RE-READ WITH ROUND-TRIP FLOAT PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 13. XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- XPT -> CSV NUMERIC FIDELITY ---")


numeric_failures = []


for col in source_df.columns:

    expected_values = (
        source_df[col]
        .to_numpy()
    )

    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    if not np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        numeric_failures.append(
            col
        )


if numeric_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(numeric_failures)
    )


print(
    "PASS: Both columns numerically match "
    "the original XPT exactly."
)


# ============================================================
# 14. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in source_df.columns:

    source_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )

    csv_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            source_missing
            !=
            csv_missing
        )
    )


print(
    "\nMissing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 15. READ RAW CSV
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)
    raw_rows = list(reader)


# ============================================================
# 16. SEQN RAW FORMAT CHECK
# ============================================================

bad_seqn_dot_zero = 0
bad_seqn_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    value = row["SEQN"]


    if value.endswith(".0"):

        bad_seqn_dot_zero += 1


        if len(
            bad_seqn_examples
        ) < 20:

            bad_seqn_examples.append(
                (
                    row_number,
                    value
                )
            )


print(
    "\nSEQN values ending '.0':",
    bad_seqn_dot_zero
)


if bad_seqn_dot_zero != 0:

    print(
        bad_seqn_examples
    )

    raise ValueError(
        "SEQN contains unnecessary .0 formatting."
    )


print(
    "PASS: SEQN has clean integer formatting."
)


# ============================================================
# 17. STRICT LBXGH RAW-TEXT FIDELITY
# ============================================================

#
# This check ensures that the laboratory decimal
# appears exactly as pandas decoded it from XPT.
#
# Examples:
#
# 5.2 -> 5.2
# 6.0 -> 6.0
#
# We do NOT change 6.0 to 6.
#

lbxgh_mismatches = 0
lbxgh_examples = []


for i, row in enumerate(
    raw_rows
):

    source_value = (
        source_df.iloc[i]["LBXGH"]
    )


    expected_text = (
        ""
        if pd.isna(source_value)
        else str(
            float(source_value)
        )
    )


    actual_text = (
        row["LBXGH"]
    )


    if actual_text != expected_text:

        lbxgh_mismatches += 1


        if len(
            lbxgh_examples
        ) < 20:

            lbxgh_examples.append(
                (
                    i + 2,
                    expected_text,
                    actual_text
                )
            )


print(
    "LBXGH raw-text mismatches:",
    lbxgh_mismatches
)


if lbxgh_mismatches != 0:

    print(
        "Examples:",
        lbxgh_examples
    )

    raise ValueError(
        "LBXGH decimal formatting changed."
    )


print(
    "PASS: LBXGH appears AS IS in CSV."
)


# ============================================================
# 18. LEGITIMATE LBXGH .0 REPORT
# ============================================================

lbxgh_dot_zero = 0


for row in raw_rows:

    value = (
        row["LBXGH"]
    )


    if (
        value != ""
        and
        value.endswith(".0")
    ):

        lbxgh_dot_zero += 1


print(
    "\nLegitimate LBXGH values ending '.0':",
    lbxgh_dot_zero
)


print(
    "IMPORTANT: These are laboratory measurements "
    "and should remain .0."
)


# ============================================================
# 19. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower_value = (
            value.lower()
        )


        if (
            "e-" in lower_value
            or
            "e+" in lower_value
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        "Examples:",
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 20. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("GHB_D CONVERSION COMPLETE")
print("=" * 80)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "Tiny-value corrections:",
    0
)

print(
    "SEQN:",
    "integer formatting"
)

print(
    "LBXGH:",
    "preserved AS IS"
)

print(
    "XPT -> CSV numeric differences:",
    0
)

print(
    "Missing-position differences:",
    0
)

print(
    "Expected SEQN .0:",
    0
)

NHANES 2005-2006 GHB_D XPT -> CSV
Rows: 6,980
Columns: 2
PASS: GHB_D structure = 6,980 rows x 2 columns.

--- TINY VALUE CHECK ---
Tiny artifact occurrences: 0
PASS: No tiny-value artifact exists.

True numeric zeros: 0
PASS: No numeric zero values in source.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 31129.0
Maximum: 41474.0
PASS: SEQN validated.

--- LBXGH ---
Valid: 6493
Missing: 487
Minimum: 3.8
Maximum: 15.6
PASS: LBXGH matches CDC (3.8-15.6, 6,493 valid, 487 missing).

LBXGH genuine fractional values: 5853
PASS: LBXGH is correctly treated as a continuous laboratory measurement.

PASS: SEQN converted to integer format.
LBXGH remains unchanged.

--- VALUE FIDELITY BEFORE EXPORT ---
PASS: SEQN numeric values unchanged.
PASS: LBXGH values unchanged.

CSV created:
C:\Users\User\Desktop\Projects\Clinical_Neighbourhood_ML_Data_Predictor\Clinical_Neighbourhood_ML_Data_Predictor\data\2005_2006\Labs\GHB_D_corrected.csv

--- XPT -> CSV NUMERIC FIDELITY ---
PASS: Both columns numericall